# **Практическая работа №5. Введение в Python. ООП рефакторинг**

---




**Обучающийся:** *Абашин Егор Сергеевич*  



---

## **Цель работы:**



Провести рефакторинг ранее созданного Python-пакета в предыдущей практической работе, преобразовав его функциональность в соответствии с принципами объектно-ориентированного программирования. Обновленный пакет должен включать классы и методы для преобразования координат между декартовой и сферической системами координат, а также для работы с файлами.



## **Задачи:**



1. Анализ существующего пакета и определение необходимых изменений для перехода на ООП-парадигму.

2. Создание новой структуры пакета с использованием классов и модулей, соответствующих принципам ООП.

3. Реализация классов и методов для преобразования координат между декартовой и сферической системами координат.

4. Реализация классов и методов для работы с файлами, обеспечивая удобный интерфейс для чтения и записи данных.

5. Создание файла `__main__.py` с консольным интерфейсом для взаимодействия с функциональностью пакета.

6. Тестирование и проверка работоспособности обновленного пакета на примерах, подтверждение корректности реализации.



## **Демонстрация результата:**



Вставьте код каждого из ваших модулей в соответствующие ячейки ниже.

**Строку, начинающуюся на %%writefile... стирать запрещено**

### **Содержимое модуля \_\_init__.py:**

In [1]:
!mkdir geo_transform_refactired

In [2]:
%%writefile geo_transform_refactired/__init__.py

from .transformations import (
    CartesianCoordinate,
    SphericalCoordinate,
    CoordinateTransformer,
)
from .utils import AngleConverter
from .file_operations import CoordinateFileManager

__all__ = [
    "CartesianCoordinate",
    "SphericalCoordinate",
    "CoordinateTransformer",
    "AngleConverter",
    "CoordinateFileManager",
]


Writing geo_transform_refactired/__init__.py


### **Содержимое модуля transformations.py:**

In [3]:
%%writefile geo_transform_refactired/transformations.py

from dataclasses import dataclass
import math


@dataclass
class CartesianCoordinate:
    x: float
    y: float
    z: float


@dataclass
class SphericalCoordinate:
    r: float
    theta: float  # азимут (радианы)
    phi: float    # полярный (радианы)


class CoordinateTransformer:
    """
    Преобразования между декартовыми и сферическими координатами.

    Используются формулы из теории:
    r = sqrt(x^2 + y^2 + z^2)
    theta = 2 * atan2(y, x + sqrt(x^2 + y^2))
    phi = atan2(sqrt(x^2 + y^2), z)

    x = r * sin(phi) * cos(theta)
    y = r * sin(phi) * sin(theta)
    z = r * cos(phi)
    """

    @staticmethod
    def cartesian_to_spherical(p: CartesianCoordinate) -> SphericalCoordinate:
        x, y, z = p.x, p.y, p.z
        r = math.sqrt(x * x + y * y + z * z)
        rho = math.sqrt(x * x + y * y)

        theta = 2.0 * math.atan2(y, x + rho)
        phi = math.atan2(rho, z)

        return SphericalCoordinate(r=r, theta=theta, phi=phi)

    @staticmethod
    def spherical_to_cartesian(s: SphericalCoordinate) -> CartesianCoordinate:
        r, theta, phi = s.r, s.theta, s.phi
        x = r * math.sin(phi) * math.cos(theta)
        y = r * math.sin(phi) * math.sin(theta)
        z = r * math.cos(phi)
        return CartesianCoordinate(x=x, y=y, z=z)

Writing geo_transform_refactired/transformations.py


### **Содержимое модуля utils.py:**

In [ ]:
%%writefile geo_transform_refactired/utils.py

import math


class AngleConverter:
    @staticmethod
    def deg_to_rad(degrees: float) -> float:
        return degrees * math.pi / 180.0

    @staticmethod
    def rad_to_deg(radians: float) -> float:
        return radians * 180.0 / math.pi


Writing geo_transform_refactired/utils.py


### **Содержимое модуля file_operations.py:**

In [5]:
%%writefile geo_transform_refactired/file_operations.py

from dataclasses import dataclass
from typing import Iterable, Literal

from .transformations import CartesianCoordinate, SphericalCoordinate
from .utils import AngleConverter


AngleUnit = Literal["rad", "deg"]


@dataclass
class CoordinateFileManager:
    """
    Читает/пишет координаты из/в текстовый файл.

    Формат входного файла: в каждой строке 3 числа.
    Разделители: пробелы / табы / запятая / точка с запятой.
    Пустые строки и строки, начинающиеся с # — игнорируются.

    Для сферических координат можно указать единицы углов: rad или deg.
    """

    @staticmethod
    def _split_line(line: str) -> list[str]:
        for sep in [",", ";"]:
            line = line.replace(sep, " ")
        return line.split()

    def read_cartesian(self, filepath: str) -> list[CartesianCoordinate]:
        out: list[CartesianCoordinate] = []
        with open(filepath, "r", encoding="utf-8") as f:
            for line_no, raw in enumerate(f, start=1):
                s = raw.strip()
                if not s or s.startswith("#"):
                    continue
                parts = self._split_line(s)
                if len(parts) != 3:
                    raise ValueError(f"Строка {line_no}: ожидалось 3 числа, получено {len(parts)} -> {raw!r}")
                x, y, z = map(float, parts)
                out.append(CartesianCoordinate(x, y, z))
        return out

    def read_spherical(self, filepath: str, angle_unit: AngleUnit = "rad") -> list[SphericalCoordinate]:
        out: list[SphericalCoordinate] = []
        with open(filepath, "r", encoding="utf-8") as f:
            for line_no, raw in enumerate(f, start=1):
                s = raw.strip()
                if not s or s.startswith("#"):
                    continue
                parts = self._split_line(s)
                if len(parts) != 3:
                    raise ValueError(f"Строка {line_no}: ожидалось 3 числа, получено {len(parts)} -> {raw!r}")
                r, theta, phi = map(float, parts)

                if angle_unit == "deg":
                    theta = AngleConverter.deg_to_rad(theta)
                    phi = AngleConverter.deg_to_rad(phi)

                out.append(SphericalCoordinate(r=r, theta=theta, phi=phi))
        return out

    def write_cartesian(self, filepath: str, coords: Iterable[CartesianCoordinate], header: str | None = None) -> None:
        with open(filepath, "w", encoding="utf-8") as f:
            if header:
                f.write(f"# {header}\n")
            for p in coords:
                f.write(f"{p.x} {p.y} {p.z}\n")

    def write_spherical(
        self,
        filepath: str,
        coords: Iterable[SphericalCoordinate],
        header: str | None = None,
        angle_unit: AngleUnit = "rad",
    ) -> None:
        with open(filepath, "w", encoding="utf-8") as f:
            if header:
                f.write(f"# {header}\n")
            for s in coords:
                theta, phi = s.theta, s.phi
                if angle_unit == "deg":
                    theta = AngleConverter.rad_to_deg(theta)
                    phi = AngleConverter.rad_to_deg(phi)
                f.write(f"{s.r} {theta} {phi}\n")



Writing geo_transform_refactired/file_operations.py


### **Содержимое модуля \_\_main__.py:**

In [6]:
%%writefile geo_transform_refactired/__main__.py

from .transformations import CartesianCoordinate, SphericalCoordinate, CoordinateTransformer
from .utils import AngleConverter
from .file_operations import CoordinateFileManager


def _ask_float(prompt: str) -> float:
    while True:
        s = input(prompt).strip().replace(",", ".")
        try:
            return float(s)
        except ValueError:
            print("Введите число, например: 1.5")


def _ask_choice(title: str, options: dict[str, str]) -> str:
    while True:
        print(title)
        for k, v in options.items():
            print(f"  {k} — {v}")
        ans = input("Ваш выбор: ").strip()
        if ans in options:
            return ans
        print("Неверный выбор.\n")


def main() -> None:
    print("=== geo_transform (ООП) ===\n")

    action = _ask_choice(
        "Что делаем?",
        {"1": "Декартовы -> сферические", "2": "Сферические -> декартовы"},
    )

    source = _ask_choice(
        "\nОткуда берём данные?",
        {"1": "Ввод вручную (одна точка)", "2": "Чтение из файла (много строк)"},
    )

    fm = CoordinateFileManager()

    if action == "2":
        angle_unit = _ask_choice("\nЕдиницы углов theta и phi:", {"1": "радианы", "2": "градусы"})
        angle_unit = "deg" if angle_unit == "2" else "rad"

    if source == "1":
        if action == "1":
            p = CartesianCoordinate(_ask_float("x = "), _ask_float("y = "), _ask_float("z = "))
            s = CoordinateTransformer.cartesian_to_spherical(p)

            show = _ask_choice("\nПоказать углы:", {"1": "в радианах", "2": "в градусах"})
            if show == "2":
                print(
                    f"\nr={s.r}, theta={AngleConverter.rad_to_deg(s.theta)}, phi={AngleConverter.rad_to_deg(s.phi)}"
                )
            else:
                print(f"\nr={s.r}, theta={s.theta}, phi={s.phi}")

        else:
            s = SphericalCoordinate(_ask_float("r = "), _ask_float("theta = "), _ask_float("phi = "))
            if angle_unit == "deg":
                s.theta = AngleConverter.deg_to_rad(s.theta)
                s.phi = AngleConverter.deg_to_rad(s.phi)

            p = CoordinateTransformer.spherical_to_cartesian(s)
            print(f"\nx={p.x}, y={p.y}, z={p.z}")

        return

    # source == "2" (из файла)
    in_path = input("\nПуть к входному файлу: ").strip()

    if action == "1":
        cart_list = fm.read_cartesian(in_path)
        sph_list = [CoordinateTransformer.cartesian_to_spherical(p) for p in cart_list]

        out_where = _ask_choice("\nКуда вывести результат?", {"1": "на экран", "2": "в файл"})
        if out_where == "1":
            show = _ask_choice("\nПоказать углы:", {"1": "в радианах", "2": "в градусах"})
            for s in sph_list:
                if show == "2":
                    print((s.r, AngleConverter.rad_to_deg(s.theta), AngleConverter.rad_to_deg(s.phi)))
                else:
                    print((s.r, s.theta, s.phi))
        else:
            out_path = input("Путь к файлу для записи: ").strip()
            out_unit = _ask_choice("\nЗаписать углы:", {"1": "в радианах", "2": "в градусах"})
            out_unit = "deg" if out_unit == "2" else "rad"
            fm.write_spherical(out_path, sph_list, header="Результаты: cartesian -> spherical", angle_unit=out_unit)
            print(f"Готово! Записано в: {out_path}")

    else:
        sph_list = fm.read_spherical(in_path, angle_unit=angle_unit)
        cart_list = [CoordinateTransformer.spherical_to_cartesian(s) for s in sph_list]

        out_where = _ask_choice("\nКуда вывести результат?", {"1": "на экран", "2": "в файл"})
        if out_where == "1":
            for p in cart_list:
                print((p.x, p.y, p.z))
        else:
            out_path = input("Путь к файлу для записи: ").strip()
            fm.write_cartesian(out_path, cart_list, header="Результаты: spherical -> cartesian")
            print(f"Готово! Записано в: {out_path}")

    print("\n=== Конец работы ===")


if __name__ == "__main__":
    main()


Writing geo_transform_refactired/__main__.py


### **Содержимое модуля main.py (с импортом пакета и тестированием функций из него):**

In [9]:
from geo_transform_refactired import (
    CartesianCoordinate,
    CoordinateTransformer,
    AngleConverter,
    CoordinateFileManager,
)

p = CartesianCoordinate(1.0, 2.0, 3.0)
s = CoordinateTransformer.cartesian_to_spherical(p)

print("Cartesian -> Spherical (rad):", s)
print("Cartesian -> Spherical (deg):", (s.r, AngleConverter.rad_to_deg(s.theta), AngleConverter.rad_to_deg(s.phi)))

p2 = CoordinateTransformer.spherical_to_cartesian(s)
print("Back to Cartesian:", p2)

print("180 deg -> rad:", AngleConverter.deg_to_rad(180))
print("pi rad -> deg:", AngleConverter.rad_to_deg(3.141592653589793))

fm = CoordinateFileManager()
print("FileManager создан:", fm)


Cartesian -> Spherical (rad): SphericalCoordinate(r=3.7416573867739413, theta=1.1071487177940904, phi=0.6405223126794246)
Cartesian -> Spherical (deg): (3.7416573867739413, 63.43494882292201, 36.699225200489884)
Back to Cartesian: CartesianCoordinate(x=1.0000000000000002, y=2.0, z=3.0)
180 deg -> rad: 3.141592653589793
pi rad -> deg: 180.0
FileManager создан: CoordinateFileManager()


In [10]:
from geo_transform.__main__ import main
main()


=== geo_transform: преобразование координат ===

Что делаем?
  1 — Декартовы -> сферические
  2 — Сферические -> декартовы

Как вводим данные?
  1 — Вручную (одна тройка)
  2 — Из файла (много строк по 3 числа)

Показать углы:
  1 — в радианах
  2 — в градусах

(r, theta, phi) = (5.196152422706632, 0.7853981633974484, 0.9553166181245092)

=== Конец работы ===
